In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('3_2').getOrCreate()

In [2]:
prices = spark.read \
            .option('header', True) \
            .option('inferSchema', True) \
            .csv('prices.csv')

In [41]:
from pyspark.sql.functions import to_date, col, desc, row_number
from pyspark.sql import Window
rownum_window = (
    Window.partitionBy('ticker', 'date').orderBy(desc(col('timestamp')))
)
prices_cleaned = prices.withColumn('date', to_date(col('timestamp'))) \
    .withColumn('price', col('price').cast('float')) \
    .filter(col('price').isNotNull()) \
    .withColumn('row_number', row_number().over(rownum_window))

In [42]:
from pyspark.sql.functions import sum, unix_timestamp, round, max
rolling_window = (
    Window.partitionBy('ticker').orderBy(unix_timestamp(col('date'))).rangeBetween(-7*86400, Window.currentRow)
)

closing_df = prices_cleaned \
    .filter(col('row_number') == 1) \
    .withColumn('closing_rolling_sum', max('price').over(rolling_window))

In [43]:
prices_cleaned = prices_cleaned.groupBy('ticker', 'date') \
.agg(round((sum((col('volume')*col('price')))/sum(col('volume'))),2).alias('vwap')) \
.fillna(0, subset=['vwap'])

In [44]:
prices_cleaned.join(closing_df, ['ticker', 'date'], how='inner').show()

+------+----------+------+-------------------+-----+------+----------+-------------------+
|ticker|      date|  vwap|          timestamp|price|volume|row_number|closing_rolling_sum|
+------+----------+------+-------------------+-----+------+----------+-------------------+
|  AAPL|2023-10-01|150.17|2023-10-01 09:30:00|150.0|  1000|         1|              150.0|
|  AAPL|2023-10-05| 155.0|2023-10-05 09:30:00|155.0|  1500|         1|              155.0|
|  TSLA|2023-10-01|   0.0|2023-10-01 11:00:00|250.0|     0|         1|              250.0|
|  TSLA|2023-10-02| 240.0|2023-10-02 10:00:00|240.0|  1000|         1|              250.0|
+------+----------+------+-------------------+-----+------+----------+-------------------+

